# 🌳 Decision Tree — Model yang Berpikir Seperti Diagram Alur

**Modul 1: Machine Learning Fundamentals** | Notebook 3 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **Decision Tree** dan bagaimana cara kerjanya
2. Bagaimana **memvisualisasi** pohon keputusan
3. Masalah **overfitting** dan cara mengatasinya
4. Teknik **hyperparameter tuning** untuk menemukan model terbaik *(bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi Sederhana

Pernahkah kamu bermain **game tebak-tebakan (20 Questions)?**

```
Pertanyaan 1: Apakah umurnya di atas 30 tahun?
  ├── Ya → Pertanyaan 2: Apakah gajinya di atas 50 juta?
  │         ├── Ya → Kemungkinan besar BELI ✅
  │         └── Tidak → Kemungkinan TIDAK beli ❌
  └── Tidak → Pertanyaan 2: Apakah gajinya di atas 80 juta?
              ├── Ya → Kemungkinan BELI ✅
              └── Tidak → Kemungkinan TIDAK beli ❌
```

**Decision Tree** bekerja persis seperti ini! Model membuat **serangkaian pertanyaan ya/tidak** tentang data, dan setiap jawaban mengarahkan ke keputusan akhir.

Ini adalah model ML yang paling **mudah dipahami** karena kita bisa melihat langsung "logika berpikir" model.

---

## 1️⃣ Persiapan — Import Library

In [ ]:
# Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Pengaturan tampilan
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Semua library berhasil di-import!')

---

## 2️⃣ Memuat Data

### 📋 Tentang Dataset

Kita menggunakan dataset **Social Network Ads** — data pengguna media sosial yang ditawari sebuah produk.

| Kolom | Keterangan |
|-------|------------|
| `Age` | Umur pengguna |
| `EstimatedSalary` | Perkiraan gaji per tahun (dollar) |
| `Purchased` | **TARGET** — Membeli produk? (1 = Ya, 0 = Tidak) |

**Pertanyaan kita:** Berdasarkan **umur** dan **gaji**, bisakah kita prediksi siapa yang akan **membeli** produk ini?

> 💡 **Tanpa upload manual?** Dataset ini juga tersedia di repo GitHub dan bisa dimuat otomatis — lihat baris komentar *Alternatif* pada cell kode di bawah (`pd.read_csv(<raw_url>)`).

In [ ]:
# Upload file
from google.colab import files
uploaded = files.upload()


# === Alternatif tanpa upload manual: muat langsung dari GitHub repo ===
# import pandas as pd
# df = pd.read_csv('https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/master/01_machine_learning_fundamentals/Social_Network_Ads.csv')


In [ ]:
# Baca dataset
data = pd.read_csv('Social_Network_Ads.csv')

print(f'Dataset: {data.shape[0]} baris, {data.shape[1]} kolom')
print(f'Kolom: {list(data.columns)}')
data.head(10)

### 📊 Eksplorasi Data

In [ ]:
# Visualisasi: siapa yang beli dan siapa yang tidak?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot — ini yang akan dipelajari Decision Tree
colors = ['#EF5350' if p == 0 else '#66BB6A' for p in data['Purchased']]
axes[0].scatter(data['Age'], data['EstimatedSalary'], c=colors, alpha=0.6, edgecolors='white')
axes[0].set_xlabel('Umur', fontsize=11)
axes[0].set_ylabel('Perkiraan Gaji ($)', fontsize=11)
axes[0].set_title('Siapa yang Membeli? (🔴 Tidak, 🟢 Ya)', fontsize=13, fontweight='bold')

# Distribusi target
jumlah = data['Purchased'].value_counts()
axes[1].pie(jumlah, labels=['Tidak Beli', 'Beli'],
            colors=['#EF5350', '#66BB6A'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporsi Pembelian', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Tidak beli: {jumlah[0]} orang ({jumlah[0]/len(data)*100:.1f}%)')
print(f'Beli:       {jumlah[1]} orang ({jumlah[1]/len(data)*100:.1f}%)')

**Pengamatan dari scatter plot:**
- Orang yang **lebih tua** dan **bergaji lebih tinggi** cenderung membeli (titik hijau di kanan atas)
- Orang **muda dengan gaji rendah** cenderung tidak membeli (titik merah di kiri bawah)
- Tapi ada juga yang gajinya tinggi tapi tidak beli, dan sebaliknya — ini yang membuat masalah menarik!

Decision Tree akan mencoba membuat **garis-garis batas** untuk memisahkan pembeli dari non-pembeli.

---

## 3️⃣ Persiapan Data

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = data.drop(columns='Purchased')
y = data['Purchased']

# Bagi data: 80% latihan, 20% ujian
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Data latihan: {X_train.shape[0]} baris')
print(f'Data ujian:   {X_test.shape[0]} baris')

---

## 4️⃣ Melatih Decision Tree (Tanpa Batasan)

Pertama, kita latih Decision Tree **tanpa batasan** apapun — biarkan pohon tumbuh sebebas mungkin.

In [ ]:
# Decision Tree tanpa batasan
tree_full = DecisionTreeClassifier(random_state=0)
tree_full.fit(X_train_scaled, y_train)

# Evaluasi
y_pred_train = tree_full.predict(X_train_scaled)
y_pred_test = tree_full.predict(X_test_scaled)

f1_train = f1_score(y_train, y_pred_train)
f1_test = f1_score(y_test, y_pred_test)

print('📊 Decision Tree TANPA Batasan')
print('=' * 40)
print(f'  F1-Score (data latihan): {f1_train:.3f} ({f1_train*100:.1f}%)')
print(f'  F1-Score (data ujian):   {f1_test:.3f} ({f1_test*100:.1f}%)')
print(f'  Kedalaman pohon:         {tree_full.get_depth()}')
print(f'  Jumlah daun:             {tree_full.get_n_leaves()}')
print('=' * 40)

### ⚠️ Masalah: Overfitting!

Perhatikan hasilnya:
- F1-Score di **data latihan** mendekati 100% (hampir sempurna!)
- F1-Score di **data ujian** jauh lebih rendah

Ini namanya **OVERFITTING** — model terlalu "menghapal" data latihan, sehingga tidak bisa menebak data baru dengan baik.

**Analogi:** Bayangkan siswa yang menghapal semua jawaban soal latihan, tapi saat ujian soalnya agak berbeda — dia kebingungan. Beda dengan siswa yang **memahami konsepnya** — dia bisa menjawab soal dalam bentuk apapun.

### Visualisasi Pohon yang Overfitting

In [ ]:
# Visualisasi pohon penuh (terlalu kompleks!)
plt.figure(figsize=(20, 10))
plot_tree(tree_full, feature_names=['Age', 'Salary'], class_names=['Tidak Beli', 'Beli'],
          filled=True, rounded=True, fontsize=7, max_depth=3)  # Tampilkan 3 level saja
plt.title(f'Decision Tree Tanpa Batasan (hanya 3 level pertama dari {tree_full.get_depth()} level)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'⚠️ Pohon ini memiliki {tree_full.get_depth()} level dan {tree_full.get_n_leaves()} daun — terlalu kompleks!')
print('Model menghapal setiap detail kecil dari data latihan.')

---

## 5️⃣ Mengatasi Overfitting: Pruning (Pemangkasan)

Solusinya: **batasi pertumbuhan pohon** (pruning). Kita bisa atur:

| Parameter | Fungsi | Analogi |
|-----------|--------|---------|
| `max_depth` | Batas kedalaman pohon | Batasi jumlah pertanyaan yang boleh ditanya |
| `min_samples_split` | Minimum data untuk membuat cabang baru | Jangan buat aturan dari terlalu sedikit contoh |
| `min_samples_leaf` | Minimum data di setiap daun | Setiap keputusan harus berdasarkan cukup banyak bukti |

### Eksperimen: Membandingkan Kedalaman Pohon

In [ ]:
# Coba berbagai kedalaman pohon
hasil = []
depths = range(1, 15)

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0)
    tree.fit(X_train_scaled, y_train)

    f1_tr = f1_score(y_train, tree.predict(X_train_scaled))
    f1_te = f1_score(y_test, tree.predict(X_test_scaled))
    hasil.append({'Kedalaman': d, 'F1 Train': f1_tr, 'F1 Test': f1_te})

df_hasil = pd.DataFrame(hasil)

# Visualisasi
plt.figure(figsize=(10, 6))
plt.plot(df_hasil['Kedalaman'], df_hasil['F1 Train'], 'o-', color='#2196F3',
         label='Data Latihan', linewidth=2, markersize=8)
plt.plot(df_hasil['Kedalaman'], df_hasil['F1 Test'], 's-', color='#FF6F00',
         label='Data Ujian', linewidth=2, markersize=8)

# Tandai kedalaman terbaik
best_depth = df_hasil.loc[df_hasil['F1 Test'].idxmax(), 'Kedalaman']
best_score = df_hasil['F1 Test'].max()
plt.axvline(x=best_depth, color='green', linestyle='--', alpha=0.5)
plt.annotate(f'Terbaik: depth={best_depth}\nF1={best_score:.3f}',
             xy=(best_depth, best_score), xytext=(best_depth+1.5, best_score-0.05),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=11, color='green', fontweight='bold')

plt.xlabel('Kedalaman Pohon (max_depth)', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.title('Pengaruh Kedalaman Pohon terhadap Performa', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'🏆 Kedalaman terbaik: {best_depth} (F1-Score test = {best_score:.3f})')

**Apa yang kita lihat?**
- Skor **data latihan** (biru) terus naik seiring pohon makin dalam — model makin "hapal"
- Skor **data ujian** (oranye) naik dulu, lalu **turun** — tanda overfitting!
- Ada "titik manis" (**sweet spot**) di mana skor ujian paling tinggi

### Melatih Model Optimal

In [ ]:
# Latih Decision Tree dengan kedalaman optimal
tree_optimal = DecisionTreeClassifier(
    max_depth=int(best_depth),
    random_state=0
)
tree_optimal.fit(X_train_scaled, y_train)

# Evaluasi
y_pred_test = tree_optimal.predict(X_test_scaled)
f1_opt = f1_score(y_test, y_pred_test)
acc_opt = accuracy_score(y_test, y_pred_test)

print('📊 Decision Tree Optimal')
print('=' * 40)
print(f'  Kedalaman:  {tree_optimal.get_depth()}')
print(f'  Jumlah daun: {tree_optimal.get_n_leaves()}')
print(f'  Accuracy:    {acc_opt:.3f} ({acc_opt*100:.1f}%)')
print(f'  F1-Score:    {f1_opt:.3f} ({f1_opt*100:.1f}%)')
print('=' * 40)

### 🌳 Visualisasi Pohon Optimal

Sekarang pohonnya jauh lebih **sederhana dan mudah dibaca**!

In [ ]:
# Visualisasi pohon optimal — mudah dibaca!
plt.figure(figsize=(18, 10))
plot_tree(tree_optimal,
          feature_names=['Umur (scaled)', 'Gaji (scaled)'],
          class_names=['Tidak Beli', 'Beli'],
          filled=True, rounded=True, fontsize=10,
          proportion=True)
plt.title(f'Decision Tree Optimal (kedalaman = {tree_optimal.get_depth()})',
          fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📖 Cara Membaca Pohon:')
print('  - Setiap kotak adalah satu "pertanyaan"')
print('  - "gini" = ukuran ketidakmurnian (makin kecil = makin pasti)')
print('  - "samples" = persentase data di node tersebut')
print('  - "value" = proporsi [Tidak Beli, Beli]')
print('  - Warna BIRU = cenderung Tidak Beli, ORANYE = cenderung Beli')

### Visualisasi Batas Keputusan (Decision Boundary)

Karena kita hanya punya 2 fitur (Umur dan Gaji), kita bisa menggambar **area keputusan** model di plot 2D.

In [ ]:
# Buat grid untuk decision boundary
x_min, x_max = X_train_scaled[:, 0].min() - 0.5, X_train_scaled[:, 0].max() + 0.5
y_min, y_max = X_train_scaled[:, 1].min() - 0.5, X_train_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                      np.arange(y_min, y_max, 0.02))

# Prediksi setiap titik di grid
Z = tree_optimal.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (X_plot, y_plot, title) in enumerate([
    (X_train_scaled, y_train, 'Data Latihan'),
    (X_test_scaled, y_test, 'Data Ujian')
]):
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    scatter = axes[idx].scatter(X_plot[:, 0], X_plot[:, 1],
                                c=y_plot, cmap='RdYlGn', edgecolors='white',
                                alpha=0.7, s=50)
    axes[idx].set_xlabel('Umur (scaled)', fontsize=11)
    axes[idx].set_ylabel('Gaji (scaled)', fontsize=11)
    axes[idx].set_title(f'Decision Boundary — {title}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 Area HIJAU = model prediksi "Beli", area MERAH = model prediksi "Tidak Beli"')
print('   Perhatikan bahwa Decision Tree membuat batas berupa garis-garis LURUS (kotak-kotak).')

### Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
cm_display = ConfusionMatrixDisplay(cm, display_labels=['Tidak Beli', 'Beli'])
cm_display.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Decision Tree Optimal', fontsize=13, fontweight='bold')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\n✅ Benar: {tn + tp} prediksi ({(tn+tp)/len(y_test)*100:.1f}%)')
print(f'❌ Salah: {fp + fn} prediksi ({(fp+fn)/len(y_test)*100:.1f}%)')

---

## 6️⃣ Perbandingan: Pohon Dangkal vs Dalam

Untuk lebih memahami overfitting, mari bandingkan tiga pohon sekaligus.

In [ ]:
# Bandingkan 3 pohon: terlalu dangkal, optimal, terlalu dalam
configs = [
    ('Terlalu Dangkal (depth=1)', 1),
    (f'Optimal (depth={int(best_depth)})', int(best_depth)),
    ('Terlalu Dalam (depth=15)', 15)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (nama, depth) in enumerate(configs):
    tree_temp = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree_temp.fit(X_train_scaled, y_train)

    Z = tree_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    axes[idx].scatter(X_test_scaled[:, 0], X_test_scaled[:, 1],
                      c=y_test, cmap='RdYlGn', edgecolors='white', s=50, alpha=0.7)

    f1 = f1_score(y_test, tree_temp.predict(X_test_scaled))
    axes[idx].set_title(f'{nama}\nF1={f1:.3f}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Umur (scaled)', fontsize=10)
    axes[idx].set_ylabel('Gaji (scaled)', fontsize=10)

plt.suptitle('Underfitting vs Just Right vs Overfitting', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('📖 Perhatikan:')
print('  🔴 Terlalu dangkal → Batas terlalu sederhana (UNDERFITTING)')
print('  🟢 Optimal → Batas pas, menangkap pola utama tanpa detail berlebih')
print('  🔴 Terlalu dalam → Batas terlalu rumit, mengikuti noise (OVERFITTING)')

---

## 🌟 BONUS: Hyperparameter Tuning dengan GridSearchCV

Tadi kita coba-coba `max_depth` secara manual. Bagaimana kalau ada **banyak parameter** yang ingin dicoba?

**GridSearchCV** otomatis mencoba **semua kombinasi** parameter dan menemukan yang terbaik!

**Analogi:** Seperti mencari resep kopi terbaik — kamu coba semua kombinasi:
- Jenis biji: Arabica, Robusta
- Jumlah gula: 1 sendok, 2 sendok
- Suhu air: 85°C, 90°C, 95°C
- Total kombinasi = 2 × 2 × 3 = **12 percobaan**

GridSearchCV melakukan hal yang sama untuk model ML!

In [ ]:
from sklearn.model_selection import GridSearchCV

# Tentukan parameter yang ingin dicoba
param_grid = {
    'criterion': ['gini', 'entropy'],          # Cara mengukur kemurnian
    'max_depth': [2, 3, 4, 5, 6],              # Kedalaman pohon
    'min_samples_split': [2, 4, 6, 10],        # Min data untuk cabang baru
}

total_kombinasi = 2 * 5 * 4
print(f'Total kombinasi yang akan dicoba: {total_kombinasi}')
print(f'Dengan 5-fold CV: {total_kombinasi * 5} model akan dilatih\n')

# Jalankan GridSearchCV
tree = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(tree, param_grid=param_grid, cv=5,
                    scoring='f1', verbose=1)
grid.fit(X_train_scaled, y_train)

print(f'\n🏆 Parameter Terbaik: {grid.best_params_}')
print(f'   F1-Score terbaik (rata-rata 5 fold): {grid.best_score_:.3f}')

In [ ]:
# Evaluasi model terbaik dari GridSearch
best_tree = grid.best_estimator_
y_pred_best = best_tree.predict(X_test_scaled)

print('📊 Perbandingan Hasil')
print('=' * 50)
print(f'{"Model":<30} {"Accuracy":>10} {"F1-Score":>10}')
print('-' * 50)

models = [
    ('Tree Tanpa Batasan', tree_full),
    (f'Tree Optimal (depth={int(best_depth)})', tree_optimal),
    ('Tree GridSearchCV', best_tree)
]

for nama, mdl in models:
    pred = mdl.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f'  {nama:<28} {acc:>10.3f} {f1:>10.3f}')

print('=' * 50)
print(f'\n🏆 Model terbaik GridSearch: {grid.best_params_}')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Decision Tree** membuat keputusan melalui serangkaian pertanyaan ya/tidak — seperti diagram alur
2. **Overfitting** terjadi ketika model terlalu detail menghapal data latihan → performa buruk di data baru
3. **Pruning** (membatasi kedalaman, minimum samples) mencegah overfitting
4. Ada "**sweet spot**" antara model yang terlalu sederhana (underfitting) dan terlalu kompleks (overfitting)
5. **GridSearchCV** membantu mencari kombinasi parameter terbaik secara otomatis

### Kelebihan Decision Tree

✅ **Mudah dipahami** — bisa divisualisasi sebagai pohon
✅ **Tidak perlu scaling** (meski kita tetap pakai di sini untuk konsistensi)
✅ Bisa menangani data numerik dan kategorikal

### Kelemahan Decision Tree

❌ Mudah **overfitting** jika tidak dibatasi
❌ **Tidak stabil** — perubahan kecil di data bisa menghasilkan pohon yang sangat berbeda
❌ Batas keputusannya selalu **garis lurus** (horizontal/vertikal)

💡 Kelemahan ini akan kita atasi di notebook-notebook berikutnya dengan **Ensemble Methods** (menggabungkan banyak pohon)!

---

### 🔜 Notebook Selanjutnya: K-Nearest Neighbors (KNN)
Kita akan belajar model yang memprediksi berdasarkan "siapa tetangga terdekatmu" — mirip pepatah "dimana bumi dipijak, disitu langit dijunjung"!